
# Data Inspection

Purpose:
- inspect the two extracted 2025 datasets
- validate merge-critical keys
- validate timestamps and ordering
- quantify missingness and text-quality issues
- detect bot/mod contamination
- identify thread-size constraints relevant for the LLM pipeline

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Resolve paths relative to the project root (parent of `notebooks/`).
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

COMMENTS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'wsb_comments_2025_clean_structural.csv'
SUBMISSIONS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'wsb_submissions_2025_recovered.csv'


## 1. Load data

In [2]:

comments = pd.read_csv(COMMENTS_PATH, low_memory=False)
submissions = pd.read_csv(SUBMISSIONS_PATH, low_memory=False)

print("Comments shape:", comments.shape)
print("Submissions shape:", submissions.shape)


Comments shape: (497867, 22)
Submissions shape: (14083, 21)


## 2. Basic schema compatibility

In [3]:

print("Comments columns == Submissions columns:", list(comments.columns) == list(submissions.columns))
print("\nComments columns:")
print(comments.columns.tolist())

summary = pd.DataFrame({
    'comments_non_null': comments.notna().sum(),
    'comments_dtype': comments.dtypes.astype(str),
    'submissions_non_null': submissions.notna().sum(),
    'submissions_dtype': submissions.dtypes.astype(str),
})
display(summary)


Comments columns == Submissions columns: False

Comments columns:
['id', 'created_utc', 'date_utc', 'source_type', 'subreddit', 'author', 'score', 'title', 'body_text', 'raw_text', 'permalink', 'link_id', 'parent_id', 'submission_id', 'matched_tickers', 'matched_terms', 'match_sources', 'match_count', 'is_multi_match', 'match_confidence', 'needs_context_filter', 'thread_position']


,comments_non_null,comments_dtype,submissions_non_null,submissions_dtype
author,497867,object,14083.0,object
body_text,497867,object,9849.0,object
created_utc,497867,float64,14083.0,float64
date_utc,497867,object,0.0,float64
id,497867,object,14083.0,object
is_multi_match,497867,int64,0.0,float64
link_id,497867,object,0.0,float64
match_confidence,497867,object,0.0,float64
match_count,497867,int64,0.0,float64
match_sources,497867,object,0.0,float64


## 3. Timestamp validation

In [4]:

comments = comments.copy()
submissions = submissions.copy()

comments['created_dt'] = pd.to_datetime(comments['created_utc'], unit='s', errors='coerce')
submissions['created_dt'] = pd.to_datetime(submissions['created_utc'], unit='s', errors='coerce')

print("Comments min:", comments['created_dt'].min())
print("Comments max:", comments['created_dt'].max())
print("Submissions min:", submissions['created_dt'].min())
print("Submissions max:", submissions['created_dt'].max())

print("\nSample converted comment timestamps:")
display(comments[['created_utc', 'created_dt']].head(10))

print("\nSample converted submission timestamps:")
display(submissions[['created_utc', 'created_dt']].head(10))


Comments min: 2025-01-01 00:02:17
Comments max: 2025-12-31 23:59:57
Submissions min: 2016-10-04 03:52:50
Submissions max: 2025-12-31 22:24:24

Sample converted comment timestamps:


,created_utc,created_dt
0,1.739565e+09,2025-02-14 20:35:46
1,1.751601e+09,2025-07-04 03:43:17
2,1.759518e+09,2025-10-03 18:58:27
3,1.747067e+09,2025-05-12 16:22:26
4,1.747752e+09,2025-05-20 14:37:14
5,1.739658e+09,2025-02-15 22:19:35
6,1.752267e+09,2025-07-11 20:44:31
7,1.752144e+09,2025-07-10 10:33:49
8,1.761703e+09,2025-10-29 01:53:52
9,1.761645e+09,2025-10-28 09:53:15



Sample converted submission timestamps:


,created_utc,created_dt
0,1.475553e+09,2016-10-04 03:52:50
1,1.486219e+09,2017-02-04 14:28:43
2,1.493421e+09,2017-04-28 23:03:08
3,1.544775e+09,2018-12-14 08:10:59
4,1.576246e+09,2019-12-13 14:02:28
5,1.584235e+09,2020-03-15 01:20:22
6,1.588832e+09,2020-05-07 06:17:44
7,1.600352e+09,2020-09-17 14:11:09
8,1.601492e+09,2020-09-30 18:48:09
9,1.607331e+09,2020-12-07 08:48:00


## 5. Merge-critical key checks

In [5]:

comments['submission_id'] = comments['submission_id'].astype(str)
submissions['id'] = submissions['id'].astype(str)
comments['link_id_norm'] = comments['link_id'].astype(str).str.replace(r'^t3_', '', regex=True)
comments['parent_prefix'] = comments['parent_id'].astype(str).str.extract(r'^(t\d+)_', expand=False)

checks = {
    'comments_submission_id_missing': comments['submission_id'].isna().sum(),
    'comments_link_id_missing': comments['link_id'].isna().sum(),
    'comments_link_norm_equals_submission_id': (comments['link_id_norm'] == comments['submission_id']).sum(),
    'submission_ids_unique_in_submissions': submissions['id'].is_unique,
    'ids_unique_in_comments': comments['id'].is_unique,
    'ids_unique_in_submissions': submissions['id'].is_unique,
}
checks


{'comments_submission_id_missing': np.int64(0),
 'comments_link_id_missing': np.int64(0),
 'comments_link_norm_equals_submission_id': np.int64(497867),
 'submission_ids_unique_in_submissions': True,
 'ids_unique_in_comments': True,
 'ids_unique_in_submissions': True}

## 6. Full merge coverage: does every comment find its submission in the full submissions file?

In [6]:

submission_id_set = set(submissions['id'].astype(str))
comments['submission_exists_in_full_submissions'] = comments['submission_id'].astype(str).isin(submission_id_set)

coverage = comments['submission_exists_in_full_submissions'].value_counts(dropna=False)
coverage_pct = comments['submission_exists_in_full_submissions'].value_counts(dropna=False, normalize=True).mul(100).round(4)

print("Coverage counts:")
display(coverage.to_frame('count'))

print("Coverage percentages:")
display(coverage_pct.to_frame('pct'))

failed_full_merge = comments.loc[
    ~comments['submission_exists_in_full_submissions'],
    ['id', 'submission_id', 'link_id', 'link_id_norm', 'parent_id', 'author', 'body_text']
].head(20)

print("Examples of comments that would fail the full merge:")
display(failed_full_merge)


Coverage counts:


,count
submission_exists_in_full_submissions,
True,497866
False,1


Coverage percentages:


,pct
submission_exists_in_full_submissions,
True,99.9998
False,0.0002


Examples of comments that would fail the full merge:


,id,submission_id,link_id,link_id_norm,parent_id,author,body_text
497861,nazzxr8,v0o5nl,t3_v0o5nl,v0o5nl,t3_v0o5nl,Apprehensive-Net3187,"Absolutely. The algorithm will take your money no matter if you go long or short, or even flipback and forth. You might have a chance if you happen to get in, in the same direction the algorithm i..."


## 7. Missingness in important fields

In [9]:

important_cols = ['id', 'created_utc', 'source_type', 'title', 'body_text', 'raw_text', 'link_id', 'parent_id', 'submission_id']
missing_table = pd.DataFrame({
    'comments_missing': comments[important_cols].isna().sum(),
    'submissions_missing': submissions[important_cols].isna().sum(),
})
display(missing_table)


,comments_missing,submissions_missing
id,0,0
created_utc,0,0
source_type,0,0
title,518700,0
body_text,0,2148
raw_text,0,0
link_id,0,12208
parent_id,0,12208
submission_id,0,0


## 8. Empty / deleted / removed / very short text checks

In [10]:

def text_flags(df, text_col='body_text'):
    s = df[text_col].fillna('').astype(str).str.strip()
    return pd.Series({
        'empty_string': (s == '').sum(),
        'deleted_tag': s.str.fullmatch(r'\[deleted\]', case=False).sum(),
        'removed_tag': s.str.fullmatch(r'\[removed\]', case=False).sum(),
        'very_short_le_5_chars': (s.str.len() <= 5).sum(),
    })

quality_table = pd.DataFrame({
    'comments': text_flags(comments),
    'submissions': text_flags(submissions),
})
display(quality_table)


,comments,submissions
empty_string,0,2148
deleted_tag,0,42
removed_tag,0,2453
very_short_le_5_chars,2256,2219


## 9. Recommended `submission_text` construction logic

In [20]:

def clean_submission_text(title, body):
    title = '' if pd.isna(title) else str(title).strip()
    body = '' if pd.isna(body) else str(body).strip()

    bad_body = (body == '') or (body.lower() in ['[removed]', '[deleted]'])

    if bad_body:
        return title
    return f"{title}\n\n{body}"

submission_text_preview = submissions[['id', 'title', 'body_text']].copy()
submission_text_preview['submission_text'] = submission_text_preview.apply(
    lambda row: clean_submission_text(row['title'], row['body_text']),
    axis=1
)
display(submission_text_preview.head(10))

,id,title,body_text,submission_text
0,1hqt8lk,"Tesla CEO Elon, sold 268,000 shares yesterday apparently for Charity","NEWS: Elon Musk this week gifted $108 million to charity via 268,000 shares of $TSLA stock. ""In connection with the Reporting Person's year-end tax planning, represents bona fide gifts of the Issu...","Tesla CEO Elon, sold 268,000 shares yesterday apparently for Charity\n\nNEWS: Elon Musk this week gifted $108 million to charity via 268,000 shares of $TSLA stock. ""In connection with the Reportin..."
1,1hqtha7,Oh wow I’m actually green this year 🤓,"P&L was a rollercoaster, happy that I came out green, and happy I stopped trading from start of December to take a break and transfer the gains out. Biggest profits were made trading MSTR options,...","Oh wow I’m actually green this year 🤓\n\nP&L was a rollercoaster, happy that I came out green, and happy I stopped trading from start of December to take a break and transfer the gains out. Bigges..."
2,1hqu67r,I hate this..,"I closed my TSLA short at 458 last week for 5K loss. Today, I sold OKLO at 20.33 for a loss of 5K. I am tired boss","I hate this..\n\nI closed my TSLA short at 458 last week for 5K loss. Today, I sold OKLO at 20.33 for a loss of 5K. I am tired boss"
3,1hqvwdc,Top loss porn of 2024,Who’s got it? I say the INTC guy What say you?,Top loss porn of 2024\n\nWho’s got it? I say the INTC guy What say you?
4,1hqwmzs,Tesla Stock Slips Further Ahead of EV Maker's Q4 Delivery Report,NaN,Tesla Stock Slips Further Ahead of EV Maker's Q4 Delivery Report
5,1hqxb7i,Meta,Meta puts should print huh? 🤔 cause they user engagement is the reason the stock value is high right. I mean the markets can stay irrational longer then I can stay solvent but however yolo puts ar...,Meta\n\nMeta puts should print huh? 🤔 cause they user engagement is the reason the stock value is high right. I mean the markets can stay irrational longer then I can stay solvent but however yolo...
6,1hqyrnx,When to sell Tesla puts?,I think that Elon gets distracted by trump and trump can’t deliver much more to Elon anyway. I think that spx tanks next year and the bigger they are the harder they fall. I was planning on holdin...,When to sell Tesla puts?\n\nI think that Elon gets distracted by trump and trump can’t deliver much more to Elon anyway. I think that spx tanks next year and the bigger they are the harder they fa...
7,1hqyyvl,1st Time Covered Calls NVDA,[removed],1st Time Covered Calls NVDA
8,1hr0cpr,Time to short Nvidia?,[removed],Time to short Nvidia?
9,1hr0czr,Amzn earnings calls,Should have sold these when AMZN broke $230 but didn't want to take the tax hit so I decided to hold until the 2nd hoping it would hold that price. (I'm going to school in April so I wouldn't be w...,Amzn earnings calls\n\nShould have sold these when AMZN broke $230 but didn't want to take the tax hit so I decided to hold until the 2nd hoping it would hold that price. (I'm going to school in A...


## 10. Duplicate checks

In [21]:

def dup_report(df, name):
    print(f'--- {name} ---')
    print('Duplicate id:', df.duplicated(subset=['id']).sum())
    print('Duplicate submission_id+created_utc+body_text:', df.duplicated(subset=['submission_id', 'created_utc', 'body_text']).sum())
    print('Duplicate raw_text:', df.duplicated(subset=['raw_text']).sum())
    print()

dup_report(comments, 'comments')
dup_report(submissions, 'submissions')


--- comments ---
Duplicate id: 0
Duplicate submission_id+created_utc+body_text: 27
Duplicate raw_text: 18398

--- submissions ---
Duplicate id: 0
Duplicate submission_id+created_utc+body_text: 0
Duplicate raw_text: 433



## 11. Inspect the potential composite duplicate comments

In [22]:

potential_dup_comments = comments[
    comments.duplicated(subset=['submission_id', 'created_utc', 'body_text'], keep=False)
].sort_values(['submission_id', 'created_utc', 'body_text'])

print("Potential duplicate comment rows:", potential_dup_comments.shape[0])
display(potential_dup_comments[['id', 'submission_id', 'created_utc', 'author', 'body_text', 'raw_text']].head(50))


Potential duplicate comment rows: 54


,id,submission_id,created_utc,author,body_text,raw_text
10689,m60wvf9,1hw1snk,1.736329e+09,mooooner,Bers buying puts on CVNA: 🧑‍🔬 Bers buying puts on NVDA: 𓀦,Bers buying puts on CVNA: 🧑‍🔬 Bers buying puts on NVDA: 𓀦
10690,m60wvp9,1hw1snk,1.736329e+09,mooooner,Bers buying puts on CVNA: 🧑‍🔬 Bers buying puts on NVDA: 𓀦,Bers buying puts on CVNA: 🧑‍🔬 Bers buying puts on NVDA: 𓀦
10691,m60x040,1hwfuxr,1.736329e+09,Visible_Currency2419,"Apple is my bet as well! The are expected to launch their first own 5G modem with iPhone SE, but then not including the mmWave part. Apple is the last player out implementing its own modem, why do...","Apple is my bet as well! The are expected to launch their first own 5G modem with iPhone SE, but then not including the mmWave part. Apple is the last player out implementing its own modem, why do..."
10692,m60x0e6,1hwfuxr,1.736329e+09,Visible_Currency2419,"Apple is my bet as well! The are expected to launch their first own 5G modem with iPhone SE, but then not including the mmWave part. Apple is the last player out implementing its own modem, why do...","Apple is my bet as well! The are expected to launch their first own 5G modem with iPhone SE, but then not including the mmWave part. Apple is the last player out implementing its own modem, why do..."
15566,m6wray6,1i0bgwm,1.736771e+09,Fun-Kitchen-3726,“Your contracts for 2/21 on NVDA have dropped 50% today” • ⁠Robinhood,“Your contracts for 2/21 on NVDA have dropped 50% today” • ⁠Robinhood
15567,m6wrbaw,1i0bgwm,1.736771e+09,Fun-Kitchen-3726,“Your contracts for 2/21 on NVDA have dropped 50% today” • ⁠Robinhood,“Your contracts for 2/21 on NVDA have dropped 50% today” • ⁠Robinhood
19235,m7edqp9,1i27le1,1.737001e+09,EZ_PZ_LM_SQ_ZE,Fuck I saw tsmc earnings was tomorrow but forgot. Apple better not have shit the bed for them. I’m holding Nvidia calls and my finger was over the sell trigger,Fuck I saw tsmc earnings was tomorrow but forgot. Apple better not have shit the bed for them. I’m holding Nvidia calls and my finger was over the sell trigger
19236,m7edqvg,1i27le1,1.737001e+09,EZ_PZ_LM_SQ_ZE,Fuck I saw tsmc earnings was tomorrow but forgot. Apple better not have shit the bed for them. I’m holding Nvidia calls and my finger was over the sell trigger,Fuck I saw tsmc earnings was tomorrow but forgot. Apple better not have shit the bed for them. I’m holding Nvidia calls and my finger was over the sell trigger
19237,m7edsp6,1i2ggzp,1.737001e+09,LiterallyAzzmilk,"250k isn’t bad around 120-130 since it tends to drop in that area. As far as holding, not sure how long you should hold. Nvda was and is volatile. With that said There are bigger guns and balls in...","250k isn’t bad around 120-130 since it tends to drop in that area. As far as holding, not sure how long you should hold. Nvda was and is volatile. With that said There are bigger guns and balls in..."
19238,m7edsyf,1i2ggzp,1.737001e+09,LiterallyAzzmilk,"250k isn’t bad around 120-130 since it tends to drop in that area. As far as holding, not sure how long you should hold. Nvda was and is volatile. With that said There are bigger guns and balls in...","250k isn’t bad around 120-130 since it tends to drop in that area. As far as holding, not sure how long you should hold. Nvda was and is volatile. With that said There are bigger guns and balls in..."


## 12. Bot / mod contamination

In [23]:

print("--- comments top authors ---")
display(comments['author'].value_counts().head(15))

print("--- submissions top authors ---")
display(submissions['author'].value_counts().head(15))

mod_like = comments[
    comments['author'].astype(str).str.contains(r'mod|bot|automoderator|visualmod', case=False, na=False)
]

print("Likely mod/bot comment rows:", len(mod_like))
display(mod_like[['id', 'author', 'submission_id', 'body_text']].head(20))


--- comments top authors ---


author
VisualMod                 15071
teamyg                     2879
wallstreetbets-ModTeam     2480
KittyLover-7               1773
Ok_Cry7572                 1630
SangerGRBY                 1373
NineAsiansAtSubway         1038
Illustrious95              1014
veryveryuniquename5         931
Mother-Chipmunk2778         922
Valkorion335786             917
Wowmuchrya                  854
spellbadgrammargood         852
DrummerCompetitive20        849
mysuruhuduga                848
Name: count, dtype: int64

--- submissions top authors ---


author
callsonreddit       90
[deleted]           70
azavio              61
Force_Hammer        45
Salt_Yak_3866       44
tke248              25
wanderingtofu       20
Decarz              20
s1n0d3utscht3k      20
Alone-Phase-8948    19
brokenb3ar          18
toydan              18
holypally0731       17
AleaBito            16
No-One7863          16
Name: count, dtype: int64

Likely mod/bot comment rows: 20808


,id,author,submission_id,body_text
1,m4rwd61,wallstreetbets-ModTeam,1hqqswc,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
2,m4rwe6u,wallstreetbets-ModTeam,1hqqsnv,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
3,m4rwfyc,wallstreetbets-ModTeam,1hqqksv,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
4,m4rxagy,wallstreetbets-ModTeam,1hqpor8,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
5,m4rxdno,wallstreetbets-ModTeam,1hqp4ih,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
13,m4s01f5,wallstreetbets-ModTeam,1hqrlpz,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
14,m4s02dv,wallstreetbets-ModTeam,1hqrir2,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
15,m4s04d6,wallstreetbets-ModTeam,1hqrf08,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."
31,m4s64bx,VisualMod,1hqnuul,You already have a bet going - TSLA to 400.0 before 03-Jan-2025 07:27 PM -05
45,m4sff22,wallstreetbets-ModTeam,1hqswjd,"Thanks for your submission! To keep things interesting, we want to see big gains and big losses! So we've set the following thresholds for Gain, Loss, and YOLO flaired posts: * YOLO posts must be ..."


## 13. Thread ordering inside comments

In [24]:

comments_sorted = comments.sort_values(['submission_id', 'created_utc']).copy()

def check_thread_order(df):
    issues = 0
    for _, group in df.groupby('submission_id'):
        if not group['created_utc'].is_monotonic_increasing:
            issues += 1
    return issues

print("Threads with wrong order:", check_thread_order(comments))
print("Threads with wrong order after explicit sort:", check_thread_order(comments_sorted))


Threads with wrong order: 0
Threads with wrong order after explicit sort: 0


## 14. Thread size distribution

In [25]:

thread_sizes = comments.groupby('submission_id').size().sort_values(ascending=False)
display(thread_sizes.describe())

largest_threads = thread_sizes.head(10).index.tolist()
print("Largest thread submission_ids:")
print(largest_threads)


count    16558.000000
mean        31.326247
std        167.012911
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max       3409.000000
dtype: float64

Largest thread submission_ids:
['1ib5x15', '1idkgpo', '1k5fvyp', '1id4ese', '1j8n736', '1k7h52x', '1ihsa6p', '1j7v45w', '1l3v9pt', '1icr8ht']


## 15. Manual inspection of one large thread

In [26]:

example_submission_id = largest_threads[0]

sub_row = submissions[submissions['id'].astype(str) == str(example_submission_id)]
com_rows = comments[comments['submission_id'].astype(str) == str(example_submission_id)].sort_values('created_utc')

print("SUBMISSION")
display(sub_row[['id', 'title', 'body_text', 'matched_tickers', 'needs_context_filter']].head(1))

print("COMMENTS")
display(com_rows[['id', 'created_utc', 'author', 'body_text', 'matched_tickers', 'needs_context_filter']].head(30))


SUBMISSION


,id,title,body_text,matched_tickers,needs_context_filter


COMMENTS


,id,created_utc,author,body_text,matched_tickers,needs_context_filter
34141,m9flyrt,1.737976e+09,Zealousdaddi,Was up 4K on msft calls over the weekend and now they are -500 lol,MSFT,0
34151,m9fm4ep,1.737976e+09,ismebbb,Nvda is going to be a penny stock by the end of the week at this rate ![img](emote|t5_2th52|31225),NVDA,0
34153,m9fm5lx,1.737976e+09,UnseasonedAndCrispy,I sold a nvda put on Thursday :/,NVDA,0
34154,m9fm5oh,1.737976e+09,Realistic_Record9527,What happened with nvda leverage x10?,NVDA,0
34156,m9fm6db,1.737976e+09,Trash_Panda_Trading,"8k NVDA looks like, really wanted index funds but todays the fire sale :/",NVDA,0
34158,m9fm7aa,1.737976e+09,apurimac777,I think i may grab the NVDA leap calls I've been eyeing for a while today ![img](emote|t5_2th52|12787),NVDA,0
34159,m9fm7iu,1.737976e+09,Alternative-Assist18,"playing it safe with GOOG, MSFT, AMZN.... basically all the b chips u can think of",MSFT,0
34161,m9fm85s,1.737976e+09,VisualMod,"NVDA at $132? That's like buying a yacht with holes. But hey, it's your money, not like you can't afford to lose it, right?",NVDA,0
34169,m9fmh2s,1.737976e+09,jlomohocob,Why Tesla again robust.,TSLA,0
34170,m9fmh5c,1.737976e+09,Disastrous_Battle_14,Jesus nvidia. Wake up![img](emote|t5_2th52|27421)![img](emote|t5_2th52|27421)![img](emote|t5_2th52|27421),NVDA,0


## 16. Interpretation checklist


After running the notebook, these are the decisions you should be able to justify clearly:

1. Are timestamps valid and directly interpretable as Unix seconds?
2. Do all comments have a valid submission key?
3. Do all comments find their submission in the full submissions file?
4. Are there mod/bot rows that should be dropped before LLM processing?
5. Are duplicate IDs absent?
6. Should duplicate `raw_text` rows be kept? (Usually yes.)
7. How should `submission_text` be built when `body_text` is empty / removed / deleted?
8. Are thread sizes too large for raw full-thread prompting? (Usually yes.)
9. Does the dataset support a thread-summary stage followed by row-level relevance labeling?
